# 实验4.5 昇腾香橙派部署深度学习网络实验

> **实验名称**：Lab4.5 昇腾香橙派部署深度学习网络实验
> **运行平台**：昇腾香橙派 AIPro（Ascend 310B NPU）
> **建议学时**：4 学时

## 实验导学

前四个实验分别学习了轻量化网络设计（Lab4.1）、模型量化（Lab4.2）、网络裁剪（Lab4.3）和知识蒸馏（Lab4.4）。本实验将这四种技术**融合**为一个完整的端侧部署流程，在**昇腾香橙派 AIPro** 开发板上完成从训练到推理的全链路实践。

昇腾香橙派 AIPro 是基于华为昇腾 Ascend 310B AI 处理器的边缘计算开发板，算力约 8 TOPS（INT8），功耗仅约 8W，非常适合端侧 AI 部署。但由于其资源有限（内存、算力远小于服务器级 Ascend 910B），我们需要综合运用轻量化技术将模型压缩到适合端侧运行的规模。

### 前四个实验的融合关系

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">步骤</th>
<th style="text-align: left;">来源实验</th>
<th style="text-align: left;">技术</th>
<th style="text-align: left;">作用</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">Lab4.1</td>
<td style="text-align: left;">MobileNetV2 轻量化网络</td>
<td style="text-align: left;">选择参数少的网络作为教师</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">Lab4.4</td>
<td style="text-align: left;">知识蒸馏</td>
<td style="text-align: left;">用大模型训练更小的学生模型</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">Lab4.3</td>
<td style="text-align: left;">L1 非结构化裁剪</td>
<td style="text-align: left;">裁剪学生模型中冗余权重</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">Lab4.2</td>
<td style="text-align: left;">静态 PTQ 量化</td>
<td style="text-align: left;">将 FP32 压缩为 INT8，体积降 4 倍</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">—</td>
<td style="text-align: left;">ONNX 导出 + ATC 转换</td>
<td style="text-align: left;">转为昇腾 OM 离线模型</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">—</td>
<td style="text-align: left;">ACL 推理</td>
<td style="text-align: left;">在香橙派上加载 OM 模型推理</td>
</tr>
</table>

## 学习目标

1. 理解**轻量化网络 + 知识蒸馏 + 裁剪 + 量化**的完整模型压缩流程
2. 掌握将 PyTorch 模型导出为 ONNX 并通过 ATC 转换为昇腾 OM 模型的方法
3. 学会使用 ACL（Ascend Computing Language）在香橙派上加载模型并推理
4. 体会模型压缩技术在端侧部署中的实际价值

## 实验流程

```
环境准备 → 数据准备(猫狗分类)
  → 步骤1: 训练 MobileNetV2 教师模型 (Lab4.1)
  → 步骤2: 知识蒸馏训练小型学生模型 (Lab4.4)
  → 步骤3: L1 裁剪学生模型 (Lab4.3)
  → 步骤4: 静态 PTQ 量化为 INT8 (Lab4.2)
  → 步骤5: 导出 ONNX 模型
  → 步骤6: ATC 转换为昇腾 OM 模型
  → 步骤7: 香橙派 ACL 推理部署
  → 总结与课后练习
```

> **说明**：本 Notebook 在昇腾香橙派上运行，描述主要流程和代码。所有 Python 代码和脚本放在 `code/` 目录下，图片放在 `images/` 目录下，课后练习答案放在 `answer/` 目录下。

---

## 一、昇腾香橙派 AIPro 平台介绍

### 1.1 硬件规格

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">规格</th>
</tr>
<tr>
<td style="text-align: left;">AI 处理器</td>
<td style="text-align: left;">昇腾 Ascend 310B</td>
</tr>
<tr>
<td style="text-align: left;">算力</td>
<td style="text-align: left;">8 TOPS (INT8) / 4 TFLOPS (FP16)</td>
</tr>
<tr>
<td style="text-align: left;">功耗</td>
<td style="text-align: left;">约 8W</td>
</tr>
<tr>
<td style="text-align: left;">内存</td>
<td style="text-align: left;">4GB / 8GB LPDDR4X</td>
</tr>
<tr>
<td style="text-align: left;">CPU</td>
<td style="text-align: left;">4 核 ARM Cortex A55</td>
</tr>
<tr>
<td style="text-align: left;">操作系统</td>
<td style="text-align: left;">Ubuntu 22.04 (aarch64)</td>
</tr>
</table>

### 1.2 软件环境

```
CANN Toolkit (Ascend Neural Network Computing Architecture)
├── ATC     - 模型转换工具 (ONNX/CAFIR/TFLITE -> OM)
├── ACL     - Ascend Computing Language (推理接口)
├── AIPP    - AI Pre-Processing (硬件加速预处理)
└── AMCT    - Ascend Model Compression Tool (量化工具)
```

### 1.3 部署流程全景图

```
┌─────────────────────────────────────────────────────────┐
│  服务器端 (训练)                                          │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐ │
│  │ MobileNetV2│→│ 知识蒸馏  │→│  L1裁剪   │→│ INT8量化  │ │
│  │  (Lab4.1) │  │  (Lab4.4) │  │  (Lab4.3) │  │  (Lab4.2) │ │
│  └──────────┘  └──────────┘  └──────────┘  └──────────┘ │
│         │                                              │
│         ▼                                              │
│  ┌──────────┐  ┌──────────┐                           │
│  │ ONNX导出  │→│ ATC转换   │                           │
│  │ (.onnx)  │  │ (.om)    │                           │
│  └──────────┘  └──────────┘                           │
└─────────────────────────┼───────────────────────────────┘
                          │ OM 模型
                          ▼
┌─────────────────────────────────────────────────────────┐
│  香橙派端侧 (推理)                                       │
│  ┌──────────────────────────────────┐                   │
│  │  ACL 加载 OM 模型 → 推理 → 输出   │                   │
│  └──────────────────────────────────┘                   │
└─────────────────────────────────────────────────────────┘
```

---

## 二、环境准备

首先检测香橙派的运行环境，确认 CANN 工具链和 Python 库是否就绪。

**代码位置**：`code/utils.py` 中的 `get_device()` 函数

**代码说明**：
- 检测 `torch_npu` 是否可用（昇腾 NPU 适配层）
- 检测 CANN 工具链版本
- 打印香橙派硬件信息

In [ ]:
import os
import sys
import platform

print('=== 昇腾香橙派环境检测 ===')
print(f'操作系统  : {platform.system()} {platform.machine()}')
print(f'Python    : {platform.python_version()}')

import torch
print(f'PyTorch   : {torch.__version__}')

try:
    import torch_npu
    print(f'torch_npu : {torch_npu.__version__}')
    if torch.npu.is_available():
        print(f'NPU 型号  : {torch.npu.get_device_name(0)}')
        device = torch.device('npu')
    else:
        print('NPU 不可用，使用 CPU')
        device = torch.device('cpu')
except ImportError:
    print('torch_npu 未安装')
    device = torch.device('cpu')

print(f'计算设备  : {device}')

import torchvision
print(f'torchvision: {torchvision.__version__}')

print('\n=== 代码目录结构 ===')
for f in sorted(os.listdir('./code')):
    print(f'  code/{f}')

## 三、数据准备

### 3.1 数据集介绍

本实验使用 `images` 文件夹中的 4 张图片进行猫狗分类（与 Lab4.1 相同）：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">图片</th>
<th style="text-align: left;">类别</th>
<th style="text-align: left;">标签</th>
</tr>
<tr>
<td style="text-align: left;">cat1.jpg</td>
<td style="text-align: left;">猫</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;">cat2.jpg</td>
<td style="text-align: left;">猫</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;">dog1.jpg</td>
<td style="text-align: left;">狗</td>
<td style="text-align: left;">1</td>
</tr>
<tr>
<td style="text-align: left;">dog2.jpg</td>
<td style="text-align: left;">狗</td>
<td style="text-align: left;">1</td>
</tr>
</table>

通过数据增强将每张图片扩充 50 次，得到 200 个训练样本。

**代码位置**：`code/utils.py` 中的 `CatDogDataset` 类和 `get_dataloaders()` 函数

**代码讲解**：
- `CatDogDataset`：自定义数据集类，支持数据增强倍数 `augment_times`
- `get_transforms()`：训练时使用随机裁剪、翻转、旋转、颜色抖动等增强
- `get_dataloaders()`：创建训练集（200样本）和测试集（4样本）的 DataLoader

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

image_dir = './images'
image_files = ['cat1.jpg', 'cat2.jpg', 'dog1.jpg', 'dog2.jpg']
titles = ['Cat 1 (label=0)', 'Cat 2 (label=0)', 'Dog 1 (label=1)', 'Dog 2 (label=1)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img_file, title in zip(axes, image_files, titles):
    img_path = os.path.join(image_dir, img_file)
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Cat and Dog Classification Dataset', fontsize=14)
plt.tight_layout()
os.makedirs('./output', exist_ok=True)
plt.savefig('./output/dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('数据集可视化已保存到 ./output/dataset_samples.png')

## 四、步骤1：训练 MobileNetV2 教师模型（融合 Lab4.1）

### 4.1 MobileNetV2 原理回顾（来自 Lab4.1）

MobileNetV2 由 Google 于 2018 年提出，核心创新是**深度可分离卷积 + 倒残差结构**：

- **深度可分离卷积**：将标准卷积拆分为 Depthwise（逐通道卷积）+ Pointwise（1×1 卷积），计算量减少约 8~9 倍
- **倒残差结构**：瘦→胖→瘦，先用 1×1 卷积扩展通道，在高维空间做 Depthwise，再压缩回低维
- 参数量约 3.5M，模型体积约 14MB，约为 ResNet18 的 1/3

### 4.2 为什么选 MobileNetV2 作为教师模型？

在香橙派部署场景中，我们需要一个**精度足够高但不过于庞大**的教师模型。MobileNetV2 相比 ResNet18 参数更少、推理更快，同时保持较高精度，是理想的教师模型选择。

### 4.3 代码实现

**代码位置**：`code/01_train_mobilenetv2.py`

**代码讲解**：

```python
# code/01_train_mobilenetv2.py 核心代码

from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

def build_mobilenetv2(num_classes=2):
    # 加载 ImageNet 预训练权重
    model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)
    # 替换分类层: 1000类 -> 2类(猫/狗)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model
```

1. 使用 `torchvision.models.mobilenet_v2` 加载 ImageNet 预训练权重
2. 将分类器最后一层替换为 `nn.Linear(1280, 2)`，输出 2 个类别
3. 使用迁移学习（预训练权重 + 微调）策略，在猫狗数据集上训练 5 个 epoch
4. 训练完成后保存模型到 `models/mobilenet_v2_catdog.pth`

**运行方式**：
```bash
cd code
python 01_train_mobilenetv2.py
```

**预期结果**：
- MobileNetV2 参数量: ~3,500,000
- 模型大小: ~14 MB
- 测试准确率: ~100%（4 张图片分类简单）

In [ ]:
# 查看教师模型训练脚本
print('=== code/01_train_mobilenetv2.py ===')
with open('./code/01_train_mobilenetv2.py', 'r', encoding='utf-8') as f:
    content = f.read()
print(content[:2000])
print('...(完整代码请查看文件)')

### 4.4 运行结果（昇腾香橙派实测截图）

在香橙派上执行 `python 01_train_mobilenetv2.py` 的实际运行结果：

![步骤1：训练MobileNetV2教师模型运行结果](images/result_step1_train.png)


## 五、步骤2：知识蒸馏训练学生模型（融合 Lab4.4）

### 5.1 知识蒸馏原理回顾（来自 Lab4.4）

知识蒸馏让一个**小模型（学生）**学习一个**大模型（教师）**的输出分布，从而在参数量大幅减少的同时保持接近教师的精度。

**核心概念**：

- **暗知识（Dark Knowledge）**：教师输出中非 top 类的概率分布包含类间关系信息
- **温度软化（Temperature Softmax）**：$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$，T>1 使分布平滑，暴露暗知识
- **蒸馏损失**：$L = \alpha \cdot L_{hard} + (1 - \alpha) \cdot T^2 \cdot L_{soft}$
  - $L_{hard}$：学生与真实标签的交叉熵
  - $L_{soft}$：学生软分布与教师软分布的 KL 散度
  - $T^2$：补偿温度对梯度的缩放

### 5.2 学生网络设计

为学生设计一个**极小的 CNN**，适合在香橙派 Ascend 310B 上部署：

```
输入(3×224×224)
  → Conv(3→16, 3×3) → BN → ReLU
  → Conv(16→32, 3×3) → BN → ReLU → MaxPool
  → Conv(32→64, 3×3) → BN → ReLU → MaxPool
  → AdaptiveAvgPool(1×1) → Flatten → Linear(64→2)
输出(2类 logits)
```

学生网络参数量约 **0.2M**，仅为 MobileNetV2 的约 6%，非常适合端侧部署。

### 5.3 代码实现

**代码位置**：`code/02_distillation.py` 和 `code/distillation_helper.py`

**代码讲解**：

```python
# code/02_distillation.py 核心代码

def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    # 硬标签损失: 学生 vs 真实标签
    hard_loss = F.cross_entropy(student_logits, labels)
    # 软标签损失: 学生软分布 vs 教师软分布 (KL散度)
    soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_teacher = F.softmax(teacher_logits / T, dim=1)
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (T * T)
    # 总损失: 加权组合
    total_loss = alpha * hard_loss + (1.0 - alpha) * soft_loss
    return total_loss, hard_loss, soft_loss
```

蒸馏训练流程：
1. 教师网络设为 `eval()` 模式，只做推理不更新参数
2. 学生网络设为 `train()` 模式，正常前向+反向
3. 每个 batch：学生前向 → 教师前向(no_grad) → 蒸馏损失 → 反传更新学生
4. 同时训练一个无蒸馏基线学生网络作对比

**运行方式**：
```bash
cd code
python 02_distillation.py
```

**预期结果**：
- 教师准确率: ~100%
- 学生(蒸馏)准确率: 接近教师
- 学生(基线)准确率: 低于蒸馏
- 蒸馏收益: 学生(蒸馏) - 学生(基线) > 0

In [ ]:
# 查看蒸馏脚本
print('=== code/02_distillation.py (前2000字符) ===')
with open('./code/02_distillation.py', 'r', encoding='utf-8') as f:
    content = f.read()
print(content[:2000])
print('...(完整代码请查看文件)')

### 5.4 运行结果（昇腾香橙派实测截图）

在香橙派上执行 `python 02_distillation.py` 的实际运行结果：

![步骤2：知识蒸馏运行结果](images/result_step2_distill.png)


## 六、步骤3：L1 非结构化裁剪（融合 Lab4.3）

### 6.1 裁剪原理回顾（来自 Lab4.3）

深度神经网络存在大量**冗余参数**，很多权重值接近 0，对输出贡献微弱。裁剪将这些权重置零，使模型变稀疏。

**L1 非结构化裁剪**：
1. 对每层权重按绝对值排序
2. 将最小的 `amount` 比例的权重置为 0
3. `prune.remove` 将掩码永久化写入权重

**API**：`torch.nn.utils.prune.l1_unstructured(module, name='weight', amount=0.3)`

### 6.2 裁剪后微调

裁剪会损失部分精度，通过少量 epoch 微调让剩余权重重新适应任务，恢复精度。

### 6.3 代码实现

**代码位置**：`code/03_prune_model.py`

**代码讲解**：

```python
# code/03_prune_model.py 核心代码

import torch.nn.utils.prune as prune

def apply_pruning(model, amount=0.3):
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            # L1非结构化裁剪: 将最小的30%权重置零
            prune.l1_unstructured(module, name='weight', amount=amount)
            # 使裁剪永久生效
            prune.remove(module, 'weight')
```

流程：
1. 加载蒸馏训练好的学生模型
2. 对所有 Conv2d 和 Linear 层执行 L1 裁剪（amount=0.3）
3. 检查稀疏率（应接近 30%）
4. 微调 3 个 epoch 恢复精度
5. 保存到 `models/student_pruned.pth`

**运行方式**：
```bash
cd code
python 03_prune_model.py
```

> **注意**：非结构化裁剪后模型体积不变（零值仍以 FP32 存储），需配合量化才能真正减小体积。这是裁剪+量化组合的理论基础。

In [ ]:
# 查看裁剪脚本
print('=== code/03_prune_model.py (前1500字符) ===')
with open('./code/03_prune_model.py', 'r', encoding='utf-8') as f:
    content = f.read()
print(content[:1500])
print('...(完整代码请查看文件)')

### 6.4 运行结果（昇腾香橙派实测截图）

在香橙派上执行 `python 03_prune_model.py` 的实际运行结果：

![步骤3：L1裁剪运行结果](images/result_step3_prune.png)


## 七、步骤4：静态训练后量化（融合 Lab4.2）

### 7.1 量化原理回顾（来自 Lab4.2）

将模型从 FP32 压缩到 INT8，每个权重从 4 字节变为 1 字节，模型体积约降为 **1/4**。

**量化公式**：$q = \text{round}(r / s) + z$，其中 $s$ 是 scale，$z$ 是 zero_point

**静态 PTQ 三步流程**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">步骤</th>
<th style="text-align: left;">操作</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">1. prepare</td>
<td style="text-align: left;">插入 Observer</td>
<td style="text-align: left;">观测器统计流经张量的 min/max</td>
</tr>
<tr>
<td style="text-align: left;">2. 校准</td>
<td style="text-align: left;">前向传播</td>
<td style="text-align: left;">用校准数据让 Observer 收集激活值范围</td>
</tr>
<tr>
<td style="text-align: left;">3. convert</td>
<td style="text-align: left;">替换为量化模块</td>
<td style="text-align: left;">权重永久转为 INT8</td>
</tr>
</table>

### 7.2 量化后端选择

香橙派使用 ARM 架构 CPU，量化后端选择 `qnnpack`（而非 x86 的 `fbgemm`）。

### 7.3 代码实现

**代码位置**：`code/04_quantize_model.py`

**代码讲解**：

```python
# code/04_quantize_model.py 核心代码

def quantize_static(model, calibration_loader, engine='qnnpack'):
    model = model.cpu().eval()
    torch.backends.quantized.engine = engine
    model.qconfig = torch.quantization.get_default_qconfig(engine)

    # 步骤1: prepare - 插入Observer
    torch.quantization.prepare(model, inplace=True)

    # 步骤2: 校准 - 前向传播统计激活范围
    with torch.no_grad():
        for images, _ in calibration_loader:
            model(images)

    # 步骤3: convert - 转换为INT8模型
    torch.quantization.convert(model, inplace=True)
    return model
```

**运行方式**：
```bash
cd code
python 04_quantize_model.py
```

**预期结果**：
- 量化前 (FP32): ~0.8 MB
- 量化后 (INT8): ~0.2 MB（约 1/4）
- 精度损失: 通常 < 2%

In [ ]:
# 查看量化脚本
print('=== code/04_quantize_model.py (前1500字符) ===')
with open('./code/04_quantize_model.py', 'r', encoding='utf-8') as f:
    content = f.read()
print(content[:1500])
print('...(完整代码请查看文件)')

## 八、步骤5：导出 ONNX 模型

### 8.1 为什么需要 ONNX？

ONNX（Open Neural Network Exchange）是开放的模型交换格式。PyTorch 模型不能直接在昇腾 NPU 上运行，需要先转为 ONNX，再用 ATC 工具转换为昇腾专用的 OM（Offline Model）格式。

```
PyTorch (.pth) → ONNX (.onnx) → ATC → 昇腾OM (.om) → ACL推理

注意：需要安装onnx包。
 pip install onnx

```

### 8.2 代码实现

**代码位置**：`code/05_export_onnx.py`

**代码讲解**：

```python
# code/05_export_onnx.py 核心代码

def export_onnx(model, output_path, input_size=(1, 3, 224, 224)):
    model = model.cpu().eval()
    dummy_input = torch.randn(*input_size)

    torch.onnx.export(
        model, dummy_input, output_path,
        export_params=True,
        opset_version=11,       # ONNX算子集版本
        do_constant_folding=True, # 常量折叠优化
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'},  # 支持动态batch
                      'output': {0: 'batch_size'}}
    )
```

**运行方式**：
```bash
cd code
python 05_export_onnx.py
```

**输出**：`models/student_model.onnx`

### 8.3 运行结果（昇腾香橙派实测截图）

首次运行 `python 05_export_onnx.py` 时提示缺少 `onnx` 包，执行 `pip install onnx` 安装后再次运行成功：

![步骤5：导出ONNX模型运行结果](images/result_step5_onnx.png)


## 九、步骤6：ATC 转换为昇腾 OM 模型

### 9.1 ATC 工具介绍

ATC（Ascend Tensor Compiler）是昇腾 CANN 工具链中的模型转换工具，将 ONNX/Caffe/TFLite 模型编译为昇腾 NPU 可执行的 OM（Offline Model）离线模型。

### 9.2 转换命令

**代码位置**：`code/06_atc_convert.sh`

**脚本讲解**：

```bash
# code/06_atc_convert.sh 核心命令

atc --framework=5 \                    # 5=ONNX框架
    --model=./models/student_model.onnx \  # 输入ONNX模型
    --output=./models/student_model \      # 输出路径(自动加.om后缀)
    --soc_version=Ascend310B3 \           # 芯片型号: 香橙派用310B
    --input_shape="input:1,3,224,224" \   # 输入形状
    --input_format=NCHW \                 # 输入数据格式
    --output_names="output"               # 输出节点名
```

**关键参数说明**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">--framework</td>
<td style="text-align: left;">5</td>
<td style="text-align: left;">5 表示 ONNX 框架</td>
</tr>
<tr>
<td style="text-align: left;">--soc_version</td>
<td style="text-align: left;">Ascend310B3</td>
<td style="text-align: left;">香橙派 AIPro 的芯片型号</td>
</tr>
<tr>
<td style="text-align: left;">--input_shape</td>
<td style="text-align: left;">input:1,3,224,224</td>
<td style="text-align: left;">batch=1, 3通道, 224×224</td>
</tr>
<tr>
<td style="text-align: left;">--input_format</td>
<td style="text-align: left;">NCHW</td>
<td style="text-align: left;">Batch-Channel-Height-Width</td>
</tr>
</table>

**运行方式**（在安装了 CANN 的环境上）：
```bash
cd code
bash 06_atc_convert.sh
```

**输出**：`models/student_model.om`（昇腾离线模型）

### 9.3 运行结果（昇腾香橙派实测截图）

首次执行 `bash 06_atc_convert.sh` 的运行情况：

![步骤6：ATC转换首次运行](images/result_step6_atc_first.png)

成功执行 `bash 06_atc_convert.sh` 完成模型转换，输出 `student_model.om`：

![步骤6：ATC转换成功结果](images/result_step6_atc_success.png)


## 十、步骤7：香橙派 ACL 推理部署

### 10.1 ACL 推理流程

ACL（Ascend Computing Language）是昇腾 AI 计算语言库，提供模型加载和推理的 Python/C++ 接口。

```
ACL 推理流程:
1. acl.init()                    # 初始化ACL
2. acl.rt.set_device(0)          # 设置NPU设备
3. acl.mdl.load_from_file(om)    # 加载OM模型
4. 创建输入/输出dataset          # 准备数据
5. acl.mdl.execute()             # 执行推理
6. 获取输出结果                  # 后处理
7. acl.mdl.unload() / finalize() # 清理
```

### 10.2 代码实现

**代码位置**：`code/07_acl_inference.py`

**代码讲解**：

```python
# code/07_acl_inference.py 核心代码

def preprocess_image(image_path):
    # 与训练时保持一致的预处理
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert('RGB')
    return transform(img).unsqueeze(0).numpy()

# 推理
input_data = preprocess_image('../../images/cat1.jpg')
output = acl_inference('./models/student_model.om', input_data)
# 后处理: softmax + argmax
probs = softmax(output)
pred = argmax(probs)
```

**运行方式**（在香橙派上）：
```bash
cd code
python 07_acl_inference.py
```

**预期结果**：
```
图片                真实标签    预测标签    置信度      结果
cat1.jpg            cat         cat         0.98xx     OK
cat2.jpg            cat         cat         0.97xx     OK
dog1.jpg            dog         dog         0.99xx     OK
dog2.jpg            dog         dog         0.98xx     OK

推理完成: 4/4 正确, 准确率=100.0%
```

In [ ]:
# 查看ACL推理脚本
print('=== code/07_acl_inference.py (前2000字符) ===')
with open('./code/07_acl_inference.py', 'r', encoding='utf-8') as f:
    content = f.read()
print(content[:2000])
print('...(完整代码请查看文件)')

### 10.3 运行结果（昇腾香橙派实测截图）

在香橙派上执行 `python 07_acl_inference.py` 的实际推理结果：

![步骤7：ACL推理运行结果](images/result_step7_acl.png)


## 十一、完整流程总结

### 11.1 模型压缩效果对比

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">技术</th>
<th style="text-align: left;">参数量</th>
<th style="text-align: left;">模型大小</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">原始</td>
<td style="text-align: left;">MobileNetV2</td>
<td style="text-align: left;">~3.5M</td>
<td style="text-align: left;">~14 MB</td>
<td style="text-align: left;">教师模型</td>
</tr>
<tr>
<td style="text-align: left;">蒸馏后</td>
<td style="text-align: left;">StudentCNN</td>
<td style="text-align: left;">~0.2M</td>
<td style="text-align: left;">~0.8 MB</td>
<td style="text-align: left;">参数降为 6%</td>
</tr>
<tr>
<td style="text-align: left;">裁剪后</td>
<td style="text-align: left;">StudentCNN(30%稀疏)</td>
<td style="text-align: left;">~0.2M</td>
<td style="text-align: left;">~0.8 MB</td>
<td style="text-align: left;">稀疏但体积不变</td>
</tr>
<tr>
<td style="text-align: left;">量化后</td>
<td style="text-align: left;">INT8 StudentCNN</td>
<td style="text-align: left;">~0.2M</td>
<td style="text-align: left;">~0.2 MB</td>
<td style="text-align: left;">体积降为 1/4</td>
</tr>
<tr>
<td style="text-align: left;">OM模型</td>
<td style="text-align: left;">昇腾离线模型</td>
<td style="text-align: left;">—</td>
<td style="text-align: left;">~0.2 MB</td>
<td style="text-align: left;">部署到香橙派</td>
</tr>
</table>

### 11.2 各步骤代码文件一览

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">步骤</th>
<th style="text-align: left;">代码文件</th>
<th style="text-align: left;">功能</th>
</tr>
<tr>
<td style="text-align: left;">公共工具</td>
<td style="text-align: left;"><code>code/utils.py</code></td>
<td style="text-align: left;">数据加载、评估、训练函数</td>
</tr>
<tr>
<td style="text-align: left;">模型定义</td>
<td style="text-align: left;"><code>code/distillation_helper.py</code></td>
<td style="text-align: left;">StudentCNN 定义、模型加载</td>
</tr>
<tr>
<td style="text-align: left;">步骤1</td>
<td style="text-align: left;"><code>code/01_train_mobilenetv2.py</code></td>
<td style="text-align: left;">训练 MobileNetV2 教师模型</td>
</tr>
<tr>
<td style="text-align: left;">步骤2</td>
<td style="text-align: left;"><code>code/02_distillation.py</code></td>
<td style="text-align: left;">知识蒸馏训练学生模型</td>
</tr>
<tr>
<td style="text-align: left;">步骤3</td>
<td style="text-align: left;"><code>code/03_prune_model.py</code></td>
<td style="text-align: left;">L1 裁剪 + 微调</td>
</tr>
<tr>
<td style="text-align: left;">步骤4</td>
<td style="text-align: left;"><code>code/04_quantize_model.py</code></td>
<td style="text-align: left;">静态 PTQ 量化</td>
</tr>
<tr>
<td style="text-align: left;">步骤5</td>
<td style="text-align: left;"><code>code/05_export_onnx.py</code></td>
<td style="text-align: left;">导出 ONNX 模型</td>
</tr>
<tr>
<td style="text-align: left;">步骤6</td>
<td style="text-align: left;"><code>code/06_atc_convert.sh</code></td>
<td style="text-align: left;">ATC 转换为 OM 模型</td>
</tr>
<tr>
<td style="text-align: left;">步骤7</td>
<td style="text-align: left;"><code>code/07_acl_inference.py</code></td>
<td style="text-align: left;">香橙派 ACL 推理</td>
</tr>
</table>

### 11.3 完整运行顺序

```bash
cd code

# 服务器端：训练 + 压缩 + 转换
python 01_train_mobilenetv2.py    # 训练教师模型
python 02_distillation.py         # 知识蒸馏
python 03_prune_model.py          # L1裁剪
python 04_quantize_model.py       # INT8量化
python 05_export_onnx.py          # 导出ONNX
bash 06_atc_convert.sh            # ATC转换为OM

# 香橙派端：推理
python 07_acl_inference.py        # ACL推理
```

### 11.4 四种技术的融合价值

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">技术</th>
<th style="text-align: left;">来源</th>
<th style="text-align: left;">在本实验中的作用</th>
</tr>
<tr>
<td style="text-align: left;">轻量化网络</td>
<td style="text-align: left;">Lab4.1</td>
<td style="text-align: left;">MobileNetV2 作为教师，平衡精度与速度</td>
</tr>
<tr>
<td style="text-align: left;">知识蒸馏</td>
<td style="text-align: left;">Lab4.4</td>
<td style="text-align: left;">将教师知识迁移到极小的学生网络</td>
</tr>
<tr>
<td style="text-align: left;">网络裁剪</td>
<td style="text-align: left;">Lab4.3</td>
<td style="text-align: left;">裁剪学生模型冗余权重，增加稀疏性</td>
</tr>
<tr>
<td style="text-align: left;">模型量化</td>
<td style="text-align: left;">Lab4.2</td>
<td style="text-align: left;">INT8 量化，体积降为 1/4，适合端侧存储</td>
</tr>
</table>

> **核心结论**：通过**轻量化网络 + 知识蒸馏 + 裁剪 + 量化**的组合压缩，将 14MB 的 MobileNetV2 压缩到约 0.2MB 的 INT8 模型，压缩比约 **70 倍**，同时保持较高精度，成功部署到算力有限的昇腾香橙派开发板。

---

## 十二、课后练习

请根据本实验内容完成以下题目进行自测，检验你对昇腾香橙派部署深度学习网络的理解。

**第1题**（单选题）本实验中，MobileNetV2 作为教师模型的主要原因是？

- A. 参数量最大
- B. 推理速度最慢
- C. 参数较少且精度较高，平衡精度与速度
- D. 不需要训练


In [ ]:
q1 = ''  # 填入你的选项
print(f'第1题答案已记录：q1' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）知识蒸馏中温度软化（T>1）的作用是？

- A. 提高训练速度
- B. 使分布平滑，暴露暗知识（类间关系）
- C. 减少参数量
- D. 增加模型深度


In [ ]:
q2 = ''  # 填入你的选项
print(f'第2题答案已记录：q2' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）L1 非结构化裁剪的核心思想是？

- A. 删除整个通道
- B. 将绝对值最小的权重置零
- C. 降低数据精度
- D. 增加网络层数


In [ ]:
q3 = ''  # 填入你的选项
print(f'第3题答案已记录：q3' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）静态 PTQ 量化的三个步骤是？

- A. 训练→量化→验证
- B. prepare → 校准 → convert
- C. 量化→训练→转换
- D. prepare → 训练 → convert


In [ ]:
q4 = ''  # 填入你的选项
print(f'第4题答案已记录：q4' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）将 FP32 模型量化为 INT8，模型体积大约变为原来的？

- A. 1/2
- B. 1/4
- C. 1/8
- D. 不变


In [ ]:
q5 = ''  # 填入你的选项
print(f'第5题答案已记录：q5' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）昇腾香橙派部署流程中，ONNX 模型通过什么工具转换为 OM 模型？

- A. PyTorch
- B. ATC (Ascend Tensor Compiler)
- C. ACL
- D. ONNX Runtime


In [ ]:
q6 = ''  # 填入你的选项
print(f'第6题答案已记录：q6' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）在香橙派上加载 OM 模型进行推理使用什么接口？

- A. PyTorch
- B. TensorFlow
- C. ACL (Ascend Computing Language)
- D. ONNX Runtime


In [ ]:
q7 = ''  # 填入你的选项
print(f'第7题答案已记录：q7' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）非结构化裁剪后模型体积不变的原因是？

- A. 裁剪没有效果
- B. 零值仍以 FP32 存储，需稀疏格式或量化
- C. 裁剪率太低
- D. 模型太小


In [ ]:
q8 = ''  # 填入你的选项
print(f'第8题答案已记录：q8' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）蒸馏损失函数由哪两部分组成？

- A. L1 损失和 L2 损失
- B. 硬标签损失和软标签损失
- C. 训练损失和验证损失
- D. 前向损失和反向损失


In [ ]:
q9 = ''  # 填入你的选项
print(f'第9题答案已记录：q9' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）本实验的最终部署目标是？

- A. 服务器端训练
- B. 昇腾香橙派端侧推理
- C. 云端推理
- D. 模型训练加速


In [ ]:
q10 = ''  # 填入你的选项
print(f'第10题答案已记录：q10' if q10 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_07 import grade
grade(globals())

## 参考资料

- [昇腾 CANN 开发文档](https://hiascend.com/document)
- [ATC 模型转换工具使用指南](https://hiascend.com/document/detail/zh/CANNCommunityEdition)
- [ACL 推理接口文档](https://hiascend.com/document/detail/zh/CANNCommunityEdition)
- [香橙派 AIPro 官方文档](http://www.orangepi.cn/html/hardWare/computerAndMicrocontrollers/serviceAndSupport.html)
- [MobileNetV2 论文](https://arxiv.org/abs/1801.04381)（Sandler et al., 2018）
- [知识蒸馏论文](https://arxiv.org/abs/1503.02531)（Hinton et al., 2015）
- [PyTorch 量化教程](https://pytorch.org/tutorials/)
- [PyTorch 裁剪教程](https://pytorch.org/tutorials/intermediate/pruning_tutorial.html)
- [ONNX 官方文档](https://onnx.ai/)

---

> **实验完成！** 通过本实验，你学习了将轻量化网络、知识蒸馏、网络裁剪和模型量化四种技术融合，在昇腾香橙派开发板上完成深度学习模型从训练到端侧部署的完整流程。请完成课后练习检验学习效果。